In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

# Gaussian function
def gaussian(x, mu, sigma):
    return np.exp(-((x - mu)**2) / (2 * sigma**2))

# Parameters (easy to change)
sigma_central = 30.7   # sigma of central Gaussian
sigma_layer = 33       # sigma of layered Gaussians
spacing_vals = np.arange(10, 50.5, 0.5)  # spacing from 10 to 35 nm every 0.5 nm
x = np.arange(-100, 100.1, 0.1)  # x range in nm
n_layers_list = [2, 3, 4, 5, 6,7,8]   # number of component Gaussians

# Central Gaussian (normalized)
g_central_raw = gaussian(x, mu=0, sigma=sigma_central)
target_height = np.max(g_central_raw)
G_central = g_central_raw / np.max(g_central_raw) * target_height

# Results storage
results = []

# Test all combinations of spacing and number of bands
for spacing in spacing_vals:
    for n_layers in n_layers_list:
        # Determine centers for symmetric placement
        if n_layers % 2 == 1:
            # odd number (includes center at 0)
            half = (n_layers - 1) // 2
            centers = np.arange(-half, half + 1) * spacing
        else:
            # even number (symmetric around 0, offset by 0.5*spacing)
            half = n_layers // 2
            centers = (np.arange(-half, half) + 0.5) * spacing
        
        # Generate and sum layered Gaussians
        g_list_raw = [gaussian(x, mu=mu, sigma=sigma_layer) for mu in centers]
        g_sum_raw = np.sum(g_list_raw, axis=0)
        
        # Normalize sum to match central peak height
        g_sum = g_sum_raw / np.max(g_sum_raw) * target_height
        
        # Calculate metrics
        residual = g_sum - G_central
        rms_error = np.sqrt(np.mean(residual**2))
        correlation, _ = pearsonr(g_sum, G_central)
        
        # Store results
        results.append({
            'spacing': spacing,
            'n_layers': n_layers,
            'rms_error': rms_error,
            'correlation': correlation,
            'centers': centers
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Print full dataset sample
print("Full results dataset sample:")
print(results_df.head())

# Print best fits summary table
best_fits = results_df.loc[results_df.groupby('n_layers')['rms_error'].idxmin()][
    ['n_layers', 'spacing', 'rms_error', 'correlation']
].round(6).sort_values('n_layers')
print("\nBest fits (lowest RMS error for each n_layers):")
print(best_fits.to_string(index=False))

# Save results to files
output_dir = "gaussian_fit_results_ObsGFP_BW_33."
os.makedirs(output_dir, exist_ok=True)

# Full results CSV
results_df.to_csv(os.path.join(output_dir, "full_results.csv"), index=False)

# Best fits summary CSV
best_fits.to_csv(os.path.join(output_dir, "best_fits_summary.csv"), index=False)

# Pivot tables for heatmap plotting
pivot_rms = results_df.pivot(index='n_layers', columns='spacing', values='rms_error').round(6)
pivot_corr = results_df.pivot(index='n_layers', columns='spacing', values='correlation').round(6)

pivot_rms.to_csv(os.path.join(output_dir, "rms_error_pivot.csv"))
pivot_corr.to_csv(os.path.join(output_dir, "correlation_pivot.csv"))

# Save all in an Excel workbook with multiple sheets
with pd.ExcelWriter(os.path.join(output_dir, "gaussian_fits_analysis.xlsx")) as writer:
    results_df.to_excel(writer, sheet_name='Full_Results', index=False)
    best_fits.to_excel(writer, sheet_name='Best_Fits', index=False)
    pivot_rms.to_excel(writer, sheet_name='RMS_Pivot')
    pivot_corr.to_excel(writer, sheet_name='Correlation_Pivot')

print(f"\nResults saved in folder: {os.path.abspath(output_dir)}")


Full results dataset sample:
   spacing  n_layers  rms_error  correlation  \
0     10.0         2   0.038567     0.998404   
1     10.0         3   0.047523     0.997615   
2     10.0         4   0.060017     0.996269   
3     10.0         5   0.075991     0.994149   
4     10.0         6   0.095355     0.991013   

                                 centers  
0                            [-5.0, 5.0]  
1                     [-10.0, 0.0, 10.0]  
2               [-15.0, -5.0, 5.0, 15.0]  
3        [-20.0, -10.0, 0.0, 10.0, 20.0]  
4  [-25.0, -15.0, -5.0, 5.0, 15.0, 25.0]  

Best fits (lowest RMS error for each n_layers):
 n_layers  spacing  rms_error  correlation
        2     10.0   0.038567     0.998404
        3     10.0   0.047523     0.997615
        4     10.0   0.060017     0.996269
        5     10.0   0.075991     0.994149
        6     10.0   0.095355     0.991013
        7     10.0   0.117981     0.986610
        8     10.0   0.143685     0.980711

Results saved in folder: /User

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

# Gaussian function
def gaussian(x, mu, sigma):
    return np.exp(-((x - mu)**2) / (2 * sigma**2))

# Parameters (easy to change)
sigma_central = 47.8            # sigma of central Gaussian
sigma_layer_vals = np.arange(13, 40.5, 0.5)  # sigma of layered Gaussians to scan (13–35 nm)
x = np.arange(-100, 100.1, 0.1) # x range in nm

# Choose how many bands and their spacing (fixed now)
n_layers_list = [5]       # fixed numbers of bands to test
spacing = 17.5                              # choose a fixed spacing (nm); change as needed

# Central Gaussian (normalized)
g_central_raw = gaussian(x, mu=0, sigma=sigma_central)
target_height = np.max(g_central_raw)
G_central = g_central_raw / np.max(g_central_raw) * target_height

# Results storage
results = []

# Test all combinations of sigma_layer and number of bands
for n_layers in n_layers_list:
    # Determine centers for symmetric placement (fixed geometry)
    if n_layers % 2 == 1:
        # odd number (includes center at 0)
        half = (n_layers - 1) // 2
        centers = np.arange(-half, half + 1) * spacing
    else:
        # even number (symmetric around 0, offset by 0.5*spacing)
        half = n_layers // 2
        centers = (np.arange(-half, half) + 0.5) * spacing

    # For this geometry, scan sigma_layer
    for sigma_layer in sigma_layer_vals:
        # Generate and sum layered Gaussians
        g_list_raw = [gaussian(x, mu=mu, sigma=sigma_layer) for mu in centers]
        g_sum_raw = np.sum(g_list_raw, axis=0)

        # Normalize sum to match central peak height
        g_sum = g_sum_raw / np.max(g_sum_raw) * target_height

        # Calculate metrics
        residual = g_sum - G_central
        rms_error = np.sqrt(np.mean(residual**2))
        correlation, _ = pearsonr(g_sum, G_central)

        # Store results
        results.append({
            'sigma_layer': sigma_layer,
            'n_layers': n_layers,
            'rms_error': rms_error,
            'correlation': correlation,
            'spacing': spacing,
            'centers': centers
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Print full dataset sample
print("Full results dataset sample:")
print(results_df.head())

# Print best fits summary table (best sigma_layer per n_layers)
best_fits = results_df.loc[results_df.groupby('n_layers')['rms_error'].idxmin()][
    ['n_layers', 'sigma_layer', 'rms_error', 'correlation']
].round(6).sort_values('n_layers')
print("\nBest fits (lowest RMS error for each n_layers):")
print(best_fits.to_string(index=False))

# Save results to files
output_dir = "gaussian_fit_results_Zasp52_5_layer_sigma_scan"
os.makedirs(output_dir, exist_ok=True)

# Full results CSV
results_df.to_csv(os.path.join(output_dir, "full_results.csv"), index=False)

# Best fits summary CSV
best_fits.to_csv(os.path.join(output_dir, "best_fits_summary.csv"), index=False)

# Pivot tables for heatmap plotting: RMS and correlation vs sigma_layer, for each n_layers
pivot_rms = results_df.pivot(index='n_layers', columns='sigma_layer', values='rms_error').round(6)
pivot_corr = results_df.pivot(index='n_layers', columns='sigma_layer', values='correlation').round(6)

pivot_rms.to_csv(os.path.join(output_dir, "rms_error_pivot.csv"))
pivot_corr.to_csv(os.path.join(output_dir, "correlation_pivot.csv"))

# Save all in an Excel workbook with multiple sheets
with pd.ExcelWriter(os.path.join(output_dir, "gaussian_fits_analysis.xlsx")) as writer:
    results_df.to_excel(writer, sheet_name='Full_Results', index=False)
    best_fits.to_excel(writer, sheet_name='Best_Fits', index=False)
    pivot_rms.to_excel(writer, sheet_name='RMS_Pivot')
    pivot_corr.to_excel(writer, sheet_name='Correlation_Pivot')

print(f"\nResults saved in folder: {os.path.abspath(output_dir)}")


Full results dataset sample:
   sigma_layer  n_layers  rms_error  correlation  spacing  \
0         13.0         5   0.207291     0.968040     17.5   
1         13.5         5   0.203626     0.970215     17.5   
2         14.0         5   0.200055     0.972251     17.5   
3         14.5         5   0.196569     0.974154     17.5   
4         15.0         5   0.193159     0.975934     17.5   

                           centers  
0  [-35.0, -17.5, 0.0, 17.5, 35.0]  
1  [-35.0, -17.5, 0.0, 17.5, 35.0]  
2  [-35.0, -17.5, 0.0, 17.5, 35.0]  
3  [-35.0, -17.5, 0.0, 17.5, 35.0]  
4  [-35.0, -17.5, 0.0, 17.5, 35.0]  

Best fits (lowest RMS error for each n_layers):
 n_layers  sigma_layer  rms_error  correlation
        5         39.5   0.004797     0.999955

Results saved in folder: /Users/sane/gaussian_fit_results_Zasp52_5_layer_sigma_scan
